In [1]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("./learning_product_event_log.csv")

# 날짜 형식 변환
df["event_time"] = pd.to_datetime(df["event_time"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

# 데이터 구조 확인
print("데이터 크기:", df.shape)
print("\n데이터 미리보기")
print(df.head())

print("\n결측치")
print(df.isnull().sum())

print("\n이벤트 종류")
print(df["event_name"].value_counts())

print("\n고유 사용자 수:", df["user_id"].nunique())

print("\n이용 기기")
print(df["device"].value_counts())

print("\n유입 경로")
print(df["acquisition_channel"].value_counts())

print("\n강의 카테고리")
print(df["course_category"].value_counts())

print("\n데이터 기간")
print(df["event_time"].min(), "~", df["event_time"].max())

데이터 크기: (1259, 7)

데이터 미리보기
  user_id          event_time    event_name  device acquisition_channel  \
0   U0157 2026-09-01 08:54:00         visit      pc              search   
1   U0157 2026-09-01 08:58:00   view_course      pc              search   
2   U0157 2026-09-01 09:11:00   start_trial      pc              search   
3   U0157 2026-09-01 09:28:00  start_lesson      pc              search   
4   U0109 2026-09-01 09:30:00         visit  mobile            referral   

  course_category signup_date  
0        business  2026-09-01  
1        business  2026-09-01  
2        business  2026-09-01  
3        business  2026-09-01  
4            data  2026-09-01  

결측치
user_id                0
event_time             0
event_name             0
device                 0
acquisition_channel    0
course_category        0
signup_date            0
dtype: int64

이벤트 종류
event_name
visit              449
view_course        392
start_lesson       173
start_trial        132
complete_lesson     62
su

In [2]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("learning_product_event_log.csv")

# 날짜 형식 변환
df["event_time"] = pd.to_datetime(df["event_time"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

# 전체 사용자
total_users = df["user_id"].nunique()

# Acquisition: 서비스 방문
acquisition_users = df.loc[
    df["event_name"] == "visit",
    "user_id"
].nunique()

# Activation: 실제 학습 시작
activation_users = df.loc[
    df["event_name"] == "start_lesson",
    "user_id"
].nunique()

# Retention: 가입 후 정확히 7일째 재방문
visit_df = df[df["event_name"] == "visit"].copy()

visit_df["day"] = (
    visit_df["event_time"].dt.normalize()
    - visit_df["signup_date"].dt.normalize()
).dt.days

retention_users = visit_df.loc[
    visit_df["day"] == 7,
    "user_id"
].nunique()

# Revenue: 유료 구독
revenue_users = df.loc[
    df["event_name"] == "subscribe",
    "user_id"
].nunique()

# Referral: 강의 공유
referral_users = df.loc[
    df["event_name"] == "share",
    "user_id"
].nunique()

# AARRR 결과 정리
aarrr = pd.DataFrame({
    "stage": [
        "Acquisition",
        "Activation",
        "Retention",
        "Revenue",
        "Referral"
    ],
    "users": [
        acquisition_users,
        activation_users,
        retention_users,
        revenue_users,
        referral_users
    ]
})

aarrr["rate"] = (
    aarrr["users"] / total_users * 100
).round(1)

print(aarrr)

         stage  users   rate
0  Acquisition    260  100.0
1   Activation    150   57.7
2    Retention     62   23.8
3      Revenue     28   10.8
4     Referral     23    8.8


In [3]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("learning_product_event_log.csv")

# 날짜 형식 변환
df["event_time"] = pd.to_datetime(df["event_time"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

# Funnel 단계 정의
funnel_events = [
    "visit",
    "view_course",
    "start_trial",
    "subscribe"
]

funnel_names = [
    "서비스 방문",
    "강의 조회",
    "무료 체험 시작",
    "유료 구독"
]

# 각 단계의 고유 사용자 수 계산
funnel_users = []

for event in funnel_events:
    users = df.loc[
        df["event_name"] == event,
        "user_id"
    ].nunique()

    funnel_users.append(users)

# Funnel 결과 정리
funnel = pd.DataFrame({
    "stage": funnel_names,
    "users": funnel_users
})

# 이전 단계 대비 전환율
funnel["conversion_rate"] = (
    funnel["users"]
    / funnel["users"].shift(1)
    * 100
).round(1)

# 이전 단계 대비 이탈률
funnel["drop_rate"] = (
    100 - funnel["conversion_rate"]
).round(1)

# 첫 단계는 기준값으로 설정
funnel.loc[0, "conversion_rate"] = 100
funnel.loc[0, "drop_rate"] = 0

print(funnel)

      stage  users  conversion_rate  drop_rate
0    서비스 방문    260            100.0        0.0
1     강의 조회    235             90.4        9.6
2  무료 체험 시작    132             56.2       43.8
3     유료 구독     28             21.2       78.8


In [5]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("./learning_product_event_log.csv")

# 날짜 형식 변환
df["event_time"] = pd.to_datetime(df["event_time"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

# 방문 이벤트만 추출
visit_df = df[df["event_name"] == "visit"].copy()

# 가입일로부터 며칠 뒤 방문했는지 계산
visit_df["day"] = (
    visit_df["event_time"].dt.normalize()
    - visit_df["signup_date"].dt.normalize()
).dt.days

# 사용자별 가입일
users = (
    df[["user_id", "signup_date"]]
    .drop_duplicates("user_id")
    .copy()
)

# Cohort 기준 생성
users["cohort"] = users["signup_date"].dt.strftime("%Y-%m-%d")

# Cohort별 전체 사용자 수
cohort_size = (
    users.groupby("cohort")["user_id"]
    .nunique()
)

# 확인할 Retention 시점
retention_days = [1, 7, 14]

retention_result = pd.DataFrame({
    "cohort_size": cohort_size
})

# Cohort별 Retention 계산
for day in retention_days:
    retained_users = (
        visit_df[visit_df["day"] == day]
        .merge(
            users[["user_id", "cohort"]],
            on="user_id",
            how="left"
        )
        .groupby("cohort")["user_id"]
        .nunique()
    )

    retention_result[f"Day{day}"] = (
        retained_users / cohort_size * 100
    ).round(1)

retention_result = retention_result.fillna(0)

print(retention_result)

            cohort_size  Day1  Day7  Day14
cohort                                    
2026-09-01           21  14.3  23.8   19.0
2026-09-02           22  36.4  13.6   13.6
2026-09-03           22  27.3  18.2   22.7
2026-09-08           22  40.9  13.6   22.7
2026-09-09           25  36.0  40.0   16.0
2026-09-10           18  38.9  44.4   22.2
2026-09-15           19  31.6  21.1   15.8
2026-09-16           19  21.1  15.8   15.8
2026-09-17           27  37.0  25.9    3.7
2026-09-22           17  29.4  17.6    5.9
2026-09-23           24  54.2  20.8    8.3
2026-09-24           24  33.3  29.2   16.7


In [6]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("./learning_product_event_log.csv")

# 날짜 형식 변환
df["event_time"] = pd.to_datetime(df["event_time"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

# 사용자별 기본 정보
user_info = (
    df.sort_values("event_time")
    .groupby("user_id")
    .first()[["device", "acquisition_channel", "course_category"]]
    .reset_index()
)

# 무료 체험 시작 사용자
trial_users = set(
    df.loc[
        df["event_name"] == "start_trial",
        "user_id"
    ]
)

# 유료 구독 사용자
subscribe_users = set(
    df.loc[
        df["event_name"] == "subscribe",
        "user_id"
    ]
)

# 사용자별 무료 체험 / 유료 구독 여부
user_info["start_trial"] = (
    user_info["user_id"]
    .isin(trial_users)
)

user_info["subscribe"] = (
    user_info["user_id"]
    .isin(subscribe_users)
)

# Segment별 무료 체험 → 유료 구독 전환율 계산
def segment_conversion(data, column):
    result = (
        data[data["start_trial"]]
        .groupby(column)
        .agg(
            trial_users=("user_id", "nunique"),
            subscribe_users=("subscribe", "sum")
        )
    )

    result["conversion_rate"] = (
        result["subscribe_users"]
        / result["trial_users"]
        * 100
    ).round(1)

    return result

# 이용 기기별 비교
device_result = segment_conversion(
    user_info,
    "device"
)

# 유입 경로별 비교
channel_result = segment_conversion(
    user_info,
    "acquisition_channel"
)

# 강의 카테고리별 비교
category_result = segment_conversion(
    user_info,
    "course_category"
)

print("===== 이용 기기별 =====")
print(device_result)

print("\n===== 유입 경로별 =====")
print(channel_result)

print("\n===== 강의 카테고리별 =====")
print(category_result)

===== 이용 기기별 =====
        trial_users  subscribe_users  conversion_rate
device                                               
mobile           82               15             18.3
pc               50               13             26.0

===== 유입 경로별 =====
                     trial_users  subscribe_users  conversion_rate
acquisition_channel                                               
ad                            49               10             20.4
referral                      28                7             25.0
search                        55               11             20.0

===== 강의 카테고리별 =====
                 trial_users  subscribe_users  conversion_rate
course_category                                               
business                  32                5             15.6
data                      56               14             25.0
programming               44                9             20.5
